In [2]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import torch
from torch import nn

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")

In [8]:
df = pd.read_csv("Trek_Data.csv")

# Note: In the dataset, the first column name was empty, so to avoid the "Unnamed:0" column in the dataframe, we dropped it
unnamed_columns = [col for col in df.columns if str(col).startswith("Unnamed") or str(col).strip() == ""]
if unnamed_columns:
    df = df.drop(columns=unnamed_columns)

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
print(df.head(3).to_string(index=False)) # returns 383x8 shape




Loaded 383 rows and 8 columns.
                               Trek             Cost     Time Trip Grade Max Altitude     Accomodation         Best Travel Time       Contact or Book your Trip
             Everest Base Camp Trek \n$1,420     USD  16 Days   Moderate       5545 m Hotel/Guesthouse March - May & Sept - Dec https://www.nepalhikingteam.com
       Everest Base Camp Short Trek \n$1,295     USD  14 Days   Moderate       5545 m Hotel/Guesthouse March - May & Sept - Dec https://www.nepalhikingteam.com
Everest Base Camp Heli Shuttle Trek      \n$2000 USD  12 Days   Moderate       5545 m Hotel/Guesthouse March - May & Sept - Dec https://www.nepalhikingteam.com


In [ ]:
import re
def extract_cost_usd(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"\$?\s*([0-9][0-9,]*(?:\.\d+)?)\s*(?:USD)?", str(value), flags=re.IGNORECASE)
    return float(match.group(1).replace(",", "")) if match else np.nan

def extract_duration_days(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"([0-9]+(?:\.\d+)?)\s*days?", str(value), flags=re.IGNORECASE)
    return float(match.group(1)) if match else np.nan

def extract_altitude_m(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"([0-9][0-9,]*(?:\.\d+)?)\s*m\b", str(value), flags=re.IGNORECASE)
    return float(match.group(1).replace(",", "")) if match else np.nan

df["Cost_USD"] = df["Cost"].apply(extract_cost_usd)
df["Duration_Days"] = df["Time"].apply(extract_duration_days)
df["Altitude_m"] = df["Max Altitude"].apply(extract_altitude_m)

engineered_columns = ["Cost_USD", "Duration_Days", "Altitude_m", "Trip Grade"]
rows_before_cleaning = len(df)
df = df.dropna(subset=engineered_columns).reset_index(drop=True)
rows_dropped = rows_before_cleaning - len(df)

print(f"Rows dropped: {rows_dropped}") # returns 0, so all columns contain non null values
print(f"Clean rows retained: {len(df)}")
print(df[["Trek", "Cost_USD", "Duration_Days", "Altitude_m", "Trip Grade"]].head(8).to_string(index=False))


Rows dropped: 0
Clean rows retained: 383
                               Trek  Cost_USD  Duration_Days  Altitude_m Trip Grade
             Everest Base Camp Trek    1420.0           16.0      5545.0   Moderate
       Everest Base Camp Short Trek    1295.0           14.0      5545.0   Moderate
Everest Base Camp Heli Shuttle Trek    2000.0           12.0      5545.0   Moderate
        Everest Base Camp Heli Trek    3300.0           11.0      5545.0   Moderate
 Everest Base Camp Trek for Seniors    1800.0           20.0      5545.0   Moderate
            Everest Chola Pass Trek    1720.0           19.0      5545.0  Strenuous
      Gokyo Lake Renjo La Pass Trek    1450.0           16.0      5360.0   Moderate
           Everest High Passes Trek    1950.0           22.0      5545.0  Strenuous


In [10]:
continuous_features = ["Duration_Days", "Altitude_m"]

scaler = StandardScaler()
X_raw = scaler.fit_transform(df[continuous_features].to_numpy(dtype=float))

print("Standardized continuous feature matrix X_raw")
print(f"Shape: {X_raw.shape}")
print(pd.DataFrame(X_raw, columns=continuous_features).describe().round(4).to_string())

Standardized continuous feature matrix X_raw
Shape: (383, 2)
       Duration_Days  Altitude_m
count       383.0000    383.0000
mean         -0.0000      0.0000
std           1.0013      1.0013
min          -2.1082     -3.0261
25%          -0.7856     -0.4996
50%          -0.1243      0.4013
75%           0.7574      0.7598
max           2.7413      1.6646


In [11]:
X_centered = X_raw - np.mean(X_raw, axis=0)

U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

top_2_right_singular_vectors = Vt[:2].T
terrain_coordinates = X_centered @ top_2_right_singular_vectors

df["PC1"] = terrain_coordinates[:, 0]
df["PC2"] = terrain_coordinates[:, 1]

variance_ratios = (S ** 2) / np.sum(S ** 2)
variance_ratio_df = pd.DataFrame(
    {
        "Principal_Component": ["PC1", "PC2"],
        "Variance_Ratio": variance_ratios[:2],
        "Variance_Percentage": variance_ratios[:2] * 100,
    }
)

print("Variance Ratio Table: structural distribution captured by Himalayan Terrain Space")
print(variance_ratio_df.to_string(index=False, formatters={"Variance_Ratio": "{:.6f}".format, "Variance_Percentage": "{:.3f}%".format}))


Variance Ratio Table: structural distribution captured by Himalayan Terrain Space
Principal_Component Variance_Ratio Variance_Percentage
                PC1       0.766496             76.650%
                PC2       0.233504             23.350%


In [12]:
kmeans = KMeans(n_clusters=3, random_state=SEED, n_init=10)
df["Terrain_Cluster"] = kmeans.fit_predict(terrain_coordinates)

cluster_archetype_matrix = df.groupby("Terrain_Cluster")[["Duration_Days", "Altitude_m", "Cost_USD"]].mean().sort_index()

print("Cluster Archetype Matrices: mean route profile by terrain category")
print(cluster_archetype_matrix.round(2).to_string())

Cluster Archetype Matrices: mean route profile by terrain category
                 Duration_Days  Altitude_m  Cost_USD
Terrain_Cluster                                     
0                        18.97     5323.60   1533.90
1                        12.63     4685.31   1412.94
2                         9.40     2747.02   1123.42
